# Calibrate ZeroPoint on your own data

You need a table of texts with the right label for each one: a few hundred per label is a good start.
No examples of "none" are needed. ZeroPoint learns where your labels live and treats everything else as outside.

Runs on Colab's free CPU or GPU. With a few thousand texts it takes a few minutes.

In [ ]:
!pip install -q "zeropoint[monitor]" pandas matplotlib

## 1. Your data

A CSV with two columns, `text` and `label`. To try the notebook without your own file, the cell below uses a
public dataset of banking requests (77 labels) instead.

In [ ]:
import pandas as pd

CSV = None   # e.g. "my_tickets.csv"
if CSV:
    df = pd.read_csv(CSV)[["text", "label"]].dropna()
else:
    from datasets import load_dataset
    d = load_dataset("PolyAI/banking77", split="train")
    names = d.features["label"].names
    df = pd.DataFrame({"text": d["text"], "label": [names[i].replace("_", " ") for i in d["label"]]})

df = df.sample(frac=1, random_state=0).reset_index(drop=True)
test = df.iloc[: len(df) // 5]
train = df.iloc[len(df) // 5 :]
print(len(train), "texts to calibrate on,", len(test), "kept apart to check the result,", df.label.nunique(), "labels")
df.label.value_counts().head(10)

## 2. Calibrate

Options worth knowing:

- `descriptions`: a short description per label, when label names are terse (`{"BIL": "billing and invoices"}`).
- `catch_all`: labels that fit anything, such as "Other" or "General Inquiry".
- `encoder`: `"small"` (default, faster) or `"base"` (more accurate, about three times slower).
- `monitor=True`: ndt watches the learned space during training. If it collapses to very few dimensions,
  the model has stopped telling texts apart even when the loss looks fine, and you get a warning.

If the model does not beat chance on held-out texts by 10 points, calibration stops with an error instead of
handing you a broken model.

In [ ]:
from zeropoint import ZeroPoint

zp = ZeroPoint.fit(train.text.tolist(), train.label.tolist(), encoder="small", monitor=True)
zp

## 3. What ndt saw

The participation ratio is the number of directions the learned space really uses. It should rise and then
settle. A fall towards 1 or 2 means collapse.

In [ ]:
import matplotlib.pyplot as plt

t = zp.config["training"]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
h = pd.DataFrame(t["history"])
ax[0].plot(h.epoch, h.train_loss, label="training"); ax[0].plot(h.epoch, h.val_loss, label="validation")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].legend()
if t["participation_ratio"]:
    ax[1].plot(t["participation_ratio"]); ax[1].set_xlabel("sample (every 20 steps)"); ax[1].set_ylabel("participation ratio")
plt.tight_layout(); plt.show()
print(f"validation accuracy {t['validation_accuracy']:.3f} (chance {t['chance']:.3f})")

## 4. Check it on the texts kept apart

Accuracy when the right label is on the list, and how often ZeroPoint says "none" when it should not.
Then a stress test: the same texts with their right label removed from the list. A good model says "none".

In [ ]:
import numpy as np, random

labels = zp.labels
dec = zp.decide_many(test.text.tolist())
acc = np.mean([d.choice == y for d, y in zip(dec, test.label)])
false_none = np.mean([d.choice is None for d in dec])

rng = random.Random(0)
k = min(10, len(labels) - 1)
lists = [rng.sample([l for l in labels if l != y], k) for y in test.label]
dec_missing = zp.decide_many(test.text.tolist(), lists)
caught = np.mean([d.choice is None for d in dec_missing])

print(f"right label on the list:   accuracy {acc:.3f}, said 'none' {false_none:.3f} of the time")
print(f"right label removed:       said 'none' {caught:.3f} of the time")

## 5. Choose your threshold

`decision.choice` is None when "none" is the most probable outcome. If you prefer to act only when ZeroPoint
is very sure, use `p_none` directly. The table shows, for several thresholds, how many good answers you keep
and how many cases with no right answer you would still act on.

In [ ]:
pn_ok = np.array([d.p_none for d in dec])
pn_missing = np.array([d.p_none for d in dec_missing])
rows = []
for th in [0.1, 0.2, 0.3, 0.5, 0.7, 0.9]:
    rows.append({"act when p_none <": th, "answerable cases kept": (pn_ok < th).mean(),
                 "no-right-answer cases still acted on": (pn_missing < th).mean()})
pd.DataFrame(rows).round(3)

## 6. Save and use

The file holds everything: the wells, the temperature and the edge of your data. Load it anywhere with
`ZeroPoint.load(path)`; inference needs no PyTorch.

In [ ]:
zp.save("my_zeropoint.safetensors")
zp2 = ZeroPoint.load("my_zeropoint.safetensors")
zp2.decide(test.text.iloc[0])